In [15]:
import pandas as pd
import os
import re
import numpy as np
from dotenv import load_dotenv

In [16]:
load_dotenv()

True

In [17]:
# definindo os caminhos
path = os.getenv('DATASETS_PATH')
data_path = os.getenv('DATA_PATH')

# carregando dataset de clubs
df = pd.read_csv(path + '/clubs.csv')
df.head()

,club_id,club_code,name,domestic_competition_id,total_market_value,squad_size,average_age,foreigners_number,foreigners_percentage,national_team_players,stadium_name,stadium_seats,net_transfer_record,coach_name,last_season,filename,url
0,10,arminia-bielefeld,Arminia Bielefeld,L1,NaN,27,25.3,15,55.6,4,SchücoArena,26515,+€5.90m,NaN,2021,../data/raw/transfermarkt-scraper/2021/clubs.j...,https://www.transfermarkt.co.uk/arminia-bielef...
1,10004,paris-fc,Paris Football Club,FR1,NaN,31,28.8,17,54.8,9,Stade Jean Bouin,19904,€-72.30m,NaN,2025,../data/raw/transfermarkt-scraper/2025/clubs.j...,https://www.transfermarkt.co.uk/paris-fc/start...
2,10010,esporte-clube-bahia,Esporte Clube Bahia,BRA1,NaN,31,26.8,6,19.4,3,Arena Fonte Nova,47364,+€8.14m,NaN,2025,../data/raw/transfermarkt-scraper/2025/clubs.j...,https://www.transfermarkt.co.uk/esporte-clube-...
3,1003,leicester-city,Leicester City,GB1,NaN,29,25.9,17,58.6,9,King Power Stadium,32259,+€57.30m,Steve Cooper,2024,../data/raw/transfermarkt-scraper/2024/clubs.j...,https://www.transfermarkt.co.uk/leicester-city...
4,1005,us-lecce,Unione Sportiva Lecce,IT1,NaN,27,25.3,23,85.2,13,Ettore Giardiniero,31559,+€8.62m,NaN,2025,../data/raw/transfermarkt-scraper/2025/clubs.j...,https://www.transfermarkt.co.uk/us-lecce/start...


In [18]:
# vendo informações gerais do dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 796 entries, 0 to 795
Data columns (total 17 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   club_id                  796 non-null    int64  
 1   club_code                796 non-null    object 
 2   name                     796 non-null    object 
 3   domestic_competition_id  796 non-null    object 
 4   total_market_value       0 non-null      float64
 5   squad_size               796 non-null    int64  
 6   average_age              758 non-null    float64
 7   foreigners_number        796 non-null    int64  
 8   foreigners_percentage    738 non-null    float64
 9   national_team_players    796 non-null    int64  
 10  stadium_name             796 non-null    object 
 11  stadium_seats            796 non-null    int64  
 12  net_transfer_record      796 non-null    object 
 13  coach_name               90 non-null     object 
 14  last_season              7

In [19]:
# vendo nulos por coluna
df.isna().sum()

club_id                      0
club_code                    0
name                         0
domestic_competition_id      0
total_market_value         796
squad_size                   0
average_age                 38
foreigners_number            0
foreigners_percentage       58
national_team_players        0
stadium_name                 0
stadium_seats                0
net_transfer_record          0
coach_name                 706
last_season                  0
filename                     0
url                          0
dtype: int64

In [20]:
# carregando dataset de competitions
df_temp = pd.read_csv(path + '/competitions.csv')
df_temp.head()

,competition_id,competition_code,name,sub_type,type,country_id,country_name,domestic_league_code,confederation,total_clubs,url
0,A1,bundesliga,bundesliga,first_tier,domestic_league,127,Austria,A1,europa,12.0,https://www.transfermarkt.co.uk/bundesliga/sta...
1,AFAC,afc-asian-cup,afc-asian-cup,afc_asian_cup,national_team_competition,-1,NaN,NaN,asien,NaN,https://www.transfermarkt.co.uk/afc-asian-cup/...
2,AFCN,africa-cup-of-nations,africa-cup-of-nations,africa_cup_of_nations,national_team_competition,-1,NaN,NaN,afrika,NaN,https://www.transfermarkt.co.uk/africa-cup-of-...
3,ARG1,torneo-apertura,torneo-apertura,first_tier,domestic_league,9,Argentina,ARG1,amerika,30.0,https://www.transfermarkt.co.uk/torneo-apertur...
4,AUS1,a-league-men,a-league-men,first_tier,domestic_league,12,Australia,AUS1,asien,12.0,https://www.transfermarkt.co.uk/a-league-men/s...


In [21]:
# parser de strings monetárias como em float
def parse_money(val):
    if pd.isna(val) or str(val).strip() == '':
        return 0.0
    s = str(val).replace(' ', '').replace('€', '').replace('+', '')
    match = re.match(r'(-?[\d.]+)(m|bn|k)?', s, re.IGNORECASE)
    if not match:
        return 0.0
    number = float(match.group(1))
    suffix = (match.group(2) or '').lower()
    if suffix == 'bn':
        return number * 1_000_000_000
    if suffix == 'm':
        return number * 1_000_000
    if suffix == 'k':
        return number * 1_000
    return number

In [22]:
# aplicando parser em total_market_value e net_transfer_record
df['total_market_value'] = df['total_market_value'].apply(parse_money)
df['net_transfer_record'] = df['net_transfer_record'].apply(parse_money)
df[['total_market_value', 'net_transfer_record']].describe()

,total_market_value,net_transfer_record
count,796.0,7.960000e+02
mean,0.0,-7.882412e+04
std,0.0,2.906949e+07
min,0.0,-2.807300e+08
25%,0.0,-8.700000e+04
50%,0.0,3.410000e+05
75%,0.0,4.592500e+06
max,0.0,1.355700e+08


In [23]:
# criando league_tier: mapeamento de domestic_competition_id para nível numérico
league_tier_map = {
    'GB1': 1,
    'ES1': 2,
    'L1':  3,
    'IT1': 3,
    'FR1': 3,
}

# ligas first_tier da UEFA (exceto as já mapeadas acima) recebem tier 4
uefa_first_tier = df_temp[
    (df_temp['sub_type'] == 'first_tier') & (df_temp['confederation'] == 'europa')
]['competition_id'].tolist()

# define o nível da competição
def get_league_tier(comp_id):
    if pd.isna(comp_id):
        return 0
    if comp_id in league_tier_map:
        return league_tier_map[comp_id]
    if comp_id in uefa_first_tier:
        return 4
    return 5

df['league_tier'] = df['domestic_competition_id'].apply(get_league_tier)
df['league_tier'].value_counts().sort_index()

league_tier
1     37
2     33
3    106
4    425
5    195
Name: count, dtype: int64

In [24]:
# enriquecendo com confederation via join com competitions
df_temp = df_temp[['competition_id', 'confederation']].drop_duplicates()
df = df.merge(df_temp, left_on='domestic_competition_id', right_on='competition_id', how='left')
df.drop(columns=['competition_id'], inplace=True)
df['confederation'].value_counts(dropna=False)

confederation
europa     601
amerika    104
asien       71
NaN         20
Name: count, dtype: int64

In [25]:
# fallback
confederation_fallback = {
    'COL1': 'amerika',
}

df['confederation'] = df.apply(
    lambda row: confederation_fallback.get(row['domestic_competition_id'], row['confederation'])
    if pd.isna(row['confederation']) else row['confederation'],
    axis=1
)

In [26]:
# selecionando apenas as colunas relevantes para o modelo
cols_keep = [
    'club_id', 'name', 'domestic_competition_id',
    'total_market_value', 'squad_size',
    'net_transfer_record', 'league_tier', 'confederation'
]
df = df[cols_keep]
df.head()

,club_id,name,domestic_competition_id,total_market_value,squad_size,net_transfer_record,league_tier,confederation
0,10,Arminia Bielefeld,L1,0.0,27,5900000.0,3,europa
1,10004,Paris Football Club,FR1,0.0,31,-72300000.0,3,europa
2,10010,Esporte Clube Bahia,BRA1,0.0,31,8140000.0,5,amerika
3,1003,Leicester City,GB1,0.0,29,57300000.0,1,europa
4,1005,Unione Sportiva Lecce,IT1,0.0,27,8620000.0,3,europa


In [27]:
# verificando nulos restantes
df.isna().sum()

club_id                    0
name                       0
domestic_competition_id    0
total_market_value         0
squad_size                 0
net_transfer_record        0
league_tier                0
confederation              0
dtype: int64

In [28]:
# salvando clubs_processed.parquet
df.to_parquet(data_path + '/clubs_processed.parquet', index=False)